# Assignment 7: Clinical NLP with LLMs and Embeddings — Solution

## Setup

In [ ]:
%pip install -q -r requirements.txt

%reset -f

In [ ]:
import os
import json
import random
import numpy as np
from dotenv import load_dotenv

load_dotenv()
print("Setup complete!")

### Load Asclepius Notes

In [ ]:
with open("asclepius_notes.json") as f:
    asclepius = json.load(f)

print(f"Loaded {len(asclepius)} synthetic clinical notes from Asclepius")
print(f"Keys: {list(asclepius[0].keys())}")

In [ ]:
print(asclepius[0]["note"][:500] + "...")

In [ ]:
random.seed(2026)
sample = random.sample(asclepius, 4)
notes_p1 = [s["note"] for s in sample]

print(f"Selected {len(notes_p1)} notes for extraction")
for i, n in enumerate(notes_p1, 1):
    print(f"\n--- Note {i} ({len(n)} chars) ---")
    print(n[:150] + "...")

---

## Part 1: Clinical Entity Extraction

In [ ]:
from extractor import get_client, call_llm
import extractor

### `build_prompt`

In [ ]:
def build_prompt(note, few_shot=False):
    if few_shot:
        return f"""Extract structured medical data from clinical notes. Return JSON only.

Example:
Note: "72-year-old male with progressive dyspnea and orthopnea. BNP 1840 pg/mL, ejection fraction 25%.
Chest X-ray shows cardiomegaly with pulmonary edema. Started on furosemide 40mg IV, lisinopril 5mg daily,
and carvedilol 6.25mg BID. Assessment: Acute decompensated heart failure."

Output:
{{
  "diagnosis": "Acute decompensated heart failure",
  "medications": ["furosemide 40mg IV", "lisinopril 5mg daily", "carvedilol 6.25mg BID"],
  "lab_values": {{"BNP": "1840 pg/mL", "ejection_fraction": "25%"}},
  "confidence": 0.95
}}

Now extract from this note:
Note: "{note}"

Output:"""

    return f"""Extract structured medical data from the following clinical note.
Return ONLY a JSON object with exactly these fields:

{{
  "diagnosis": "primary diagnosis as a string",
  "medications": ["list of medications with doses"],
  "lab_values": {{"test_name": "value with units"}},
  "confidence": 0.0
}}

Rules:
- confidence is a float from 0.0 to 1.0
- include ALL medications mentioned with doses if given
- include ALL lab values with units
- if a field has no data, use an empty list [] or empty dict {{}}

Clinical note:
{note}"""

extractor.build_prompt = build_prompt

### `parse_json_response`

In [ ]:
def parse_json_response(text):
    # Direct parse
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass

    # Strip markdown code fences
    if "```" in text:
        start = text.find("```")
        end = text.rfind("```")
        if start != end:
            block = text[start:end]
            lines = block.split("\n")
            block = "\n".join(lines[1:])
            try:
                return json.loads(block)
            except json.JSONDecodeError:
                pass

    # Find outermost braces
    start = text.find("{")
    end = text.rfind("}")
    if start >= 0 and end > start:
        try:
            return json.loads(text[start : end + 1])
        except json.JSONDecodeError:
            pass

    return None

extractor.parse_json_response = parse_json_response

### `validate_response`

In [ ]:
def validate_response(response):
    if not isinstance(response, dict):
        return False
    required = {"diagnosis", "medications", "lab_values", "confidence"}
    return required.issubset(response.keys())

extractor.validate_response = validate_response

### `extract_entities`

In [ ]:
def extract_entities(note, few_shot=False):
    client, provider = get_client()
    prompt = build_prompt(note, few_shot=few_shot)
    raw = call_llm(prompt, provider=provider, client=client)
    parsed = parse_json_response(raw)
    if parsed is None:
        return None
    if not validate_response(parsed):
        return None
    return parsed

extractor.extract_entities = extract_entities

### Test extraction

In [ ]:
for i, note in enumerate(notes_p1, 1):
    result = extract_entities(note, few_shot=True)
    print(f"--- Note {i} ---")
    if result:
        print(json.dumps(result, indent=2))
    else:
        print("Extraction failed")
    print()

In [ ]:
# Save implementations to extractor.py for autograding (do not modify this cell)
import inspect as _insp

_parts = [
    '"""\nLLM Prompt Engineering Assignment: Clinical Entity Extraction\n\n'
    "Complete the functions below to extract structured data from clinical notes\n"
    'using LLM APIs.\n"""\n\n'
    "import json\nimport os\nfrom typing import Optional\n",
]

for _fn in [get_client, build_prompt, call_llm, extract_entities, validate_response, parse_json_response]:
    _parts.append("\n\n" + _insp.getsource(_fn))

with open("extractor.py", "w") as _f:
    _f.write("".join(_parts) + "\n")

print("Saved extractor.py")

---

## Part 2: Semantic Search

In [ ]:
from search import get_device
import search

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

_model = SentenceTransformer("all-MiniLM-L6-v2", device=get_device())
print(f"Model loaded on {get_device()}")

### `load_notes`

In [ ]:
def load_notes(filepath="clinical_notes.txt"):
    with open(filepath) as f:
        content = f.read()
    sections = content.split("## Note")
    notes = []
    for s in sections[1:]:
        text = s.split("\n", 1)
        if len(text) > 1:
            cleaned = text[1].strip()
            if cleaned:
                notes.append(cleaned)
    return notes

search.load_notes = load_notes

### `embed_notes`

In [ ]:
def embed_notes(notes):
    return _model.encode(notes)

search.embed_notes = embed_notes

### `find_similar`

In [ ]:
def find_similar(query, notes, embeddings, top_k=2):
    query_emb = _model.encode([query])
    scores = cosine_similarity(query_emb, embeddings)[0]
    ranked = sorted(
        zip(notes, scores),
        key=lambda x: x[1],
        reverse=True,
    )
    return [{"note": n, "score": float(s)} for n, s in ranked[:top_k]]

search.find_similar = find_similar

### `save_results`

In [ ]:
def save_results(results, filepath="search_results.json"):
    with open(filepath, "w") as f:
        json.dump(results, f, indent=2)

search.save_results = save_results

### Run the search pipeline

In [ ]:
notes = load_notes("clinical_notes.txt")
print(f"Loaded {len(notes)} notes")

embeddings = embed_notes(notes)
print(f"Embeddings: {embeddings.shape}")

queries = [
    "heart attack symptoms",
    "infectious disease with fever",
    "respiratory illness",
]

for q in queries:
    print(f"\nQuery: '{q}'")
    results = find_similar(q, notes, embeddings, top_k=2)
    for i, r in enumerate(results, 1):
        print(f"  {i}. (score: {r['score']:.3f}) {r['note'][:80]}...")

In [ ]:
# Save results (do not modify this cell)
save_results(
    find_similar("heart attack symptoms", notes, embeddings, top_k=2),
    "search_results.json",
)
print("Saved search_results.json")

In [ ]:
# Save implementations to search.py for autograding (do not modify this cell)
import inspect as _insp

_parts = [
    '"""\nSemantic Search Assignment: Clinical Note Search with Embeddings\n\n'
    "Use sentence embeddings to search clinical notes by meaning rather than keywords.\n"
    '"""\n\n'
    "import json\nimport numpy as np\nfrom typing import List, Dict\n",
]

_parts.append("\n\n" + _insp.getsource(get_device))
_parts.append('\n\nfrom sentence_transformers import SentenceTransformer\n')
_parts.append('from sklearn.metrics.pairwise import cosine_similarity\n\n')
_parts.append('_model = SentenceTransformer("all-MiniLM-L6-v2", device=get_device())\n')

for _fn in [load_notes, embed_notes, find_similar, save_results]:
    _parts.append("\n\n" + _insp.getsource(_fn))

with open("search.py", "w") as _f:
    _f.write("".join(_parts) + "\n")

print("Saved search.py")

---

## Validation

In [ ]:
print("Run 'pytest .github/tests/ -v' in your terminal to check your work.")